# Building a Strategy from Scratch

This notebook walks through building a **Bollinger Band Breakout** strategy using QuantCore. From an empty class to a full tearsheet with parameter analysis.

It's meant to show exactly how the library fits together when you're starting with your own idea rather than adapting an existing template.

**The idea:**  
Bollinger Bands place dynamic envelopes around a rolling mean at ±N standard deviations. When price breaks *outside* the upper band, we expect the trend to continue (momentum). When it breaks below the lower band, we go short. This is the opposite logic from mean reversion. We're betting the move keeps going, not that it snaps back.

**Contents:**
1. Environment setup
2. Generating synthetic price data
3. Building the strategy class
4. Running the first backtest
5. Performance tearsheet
6. Parameter sensitivity
7. Walk-forward analysis
8. Reflection; when does this break?

## 1. Setup

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import quantcore as qc
from quantcore.analytics import (
    calculate_all_metrics,
    calculate_returns,
    rolling_sharpe,
)
from quantcore.plotting import plot_full_tearsheet


def strip_sentinel(results: dict) -> tuple:
    """Drop the zero-timestamp sentinel the engine prepends to the equity curve."""
    equity = np.array(results['equity_curve'])
    ts     = np.array(results['timestamps'], dtype='int64')
    first_real = np.argmax(ts > 0)
    return equity[first_real:], ts[first_real:]


print(f'QuantCore {qc.version()}')

## 2. Synthetic Data

We simulate a trending price with occasional reversals. Unlike the Ornstein-Uhlenbeck process used in mean reversion, this uses a random walk with momentum (autocorrelated returns), a regime where trend-following strategies tend to do well.

In [ ]:
np.random.seed(7)

N     = 1500   # trading bars (~6 years of daily data)
sigma = 1.2    # noise per bar
rho   = 0.15   # return autocorrelation (momentum)

# Autocorrelated random walk
innovations = np.random.randn(N) * sigma
returns_sim = np.empty(N)
returns_sim[0] = innovations[0]
for i in range(1, N):
    returns_sim[i] = rho * returns_sim[i - 1] + np.sqrt(1 - rho**2) * innovations[i]

prices    = 100.0 * np.exp(np.cumsum(returns_sim / 100.0))
start_ns  = int(pd.Timestamp('2018-01-02').value)
day_ns    = int(pd.Timedelta('1D').value)
timestamps = [start_ns + i * day_ns for i in range(N)]

bars = [
    qc.BarData(
        'ASSET',
        timestamps[i],
        prices[i],
        prices[i] * (1 + abs(np.random.randn() * 0.003)),
        prices[i] * (1 - abs(np.random.randn() * 0.003)),
        prices[i],
        1_000_000.0,
    )
    for i in range(N)
]

fig, ax = plt.subplots(figsize=(14, 4))
dates = pd.to_datetime(timestamps, unit='ns')
ax.plot(dates, prices, linewidth=1.2, color='#2E86AB')
ax.set_title('Simulated Trending Price (Autocorrelated Random Walk)', fontsize=14, fontweight='bold')
ax.set_ylabel('Price ($)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Bars: {N}  |  Price range: ${prices.min():.2f} – ${prices.max():.2f}')
print(f'Autocorrelation (lag-1): {np.corrcoef(returns_sim[:-1], returns_sim[1:])[0, 1]:.3f}')

## 3. Building the Strategy

To write a custom strategy in QuantCore, subclass `qc.Strategy` and implement `on_data`. That method is called once per bar in chronological order.

Three things to keep in mind:
- `self.get_position(symbol)` returns current position size (positive = long, negative = short, 0 = flat)
- `self.generate_signal(symbol, signal_type, strength, timestamp_ns)` queues a signal for execution
- You never call the order book directly — signals flow through the engine and come back as fills via `on_fill`

The indicator math here is deliberately plain Python/NumPy, no external TA library needed.

In [ ]:
class BollingerBreakout(qc.Strategy):
    """
    Bollinger Band momentum/breakout strategy.

    Enters long when price closes above the upper band (upward breakout),
    enters short when price closes below the lower band (downward breakout).
    Exits when price crosses back through the middle band (mean).

    Parameters
    ----------
    window : int
        Lookback window for the rolling mean and standard deviation.
    n_std : float
        Number of standard deviations for the band width.
    """

    def __init__(self, window: int = 20, n_std: float = 2.0):
        super().__init__('BollingerBreakout')
        self.window = window
        self.n_std  = n_std
        self._prices = []

    def on_data(self, event: qc.MarketDataEvent):
        self._prices.append(event.get_close())

        if len(self._prices) < self.window:
            return

        window_prices = self._prices[-self.window:]
        mean = float(np.mean(window_prices))
        std  = float(np.std(window_prices, ddof=0))

        upper = mean + self.n_std * std
        lower = mean - self.n_std * std
        price = event.get_close()
        pos   = self.get_position(event.get_symbol())

        # breakout entries
        if price > upper and pos <= 0:
            self.generate_signal(event.get_symbol(), qc.SignalType.BUY,  1.0, event.get_timestamp())
        elif price < lower and pos >= 0:
            self.generate_signal(event.get_symbol(), qc.SignalType.SELL, 1.0, event.get_timestamp())
        # mean-cross exits
        elif pos > 0 and price < mean:
            self.generate_signal(event.get_symbol(), qc.SignalType.SELL, 1.0, event.get_timestamp())
        elif pos < 0 and price > mean:
            self.generate_signal(event.get_symbol(), qc.SignalType.BUY,  1.0, event.get_timestamp())

    def on_fill(self, fill: qc.FillEvent):
        pass  # nothing extra needed here

    def reset(self):
        super().reset()
        self._prices = []


# quick sanity check before running anything
s = BollingerBreakout(window=20, n_std=2.0)
print(f'Strategy name: {s.get_name()}')
print(f'Position before any data: {s.get_position("ASSET")}')

## 4. First Backtest

`qc.run_backtest` is the simplest entry point. Pass in the strategy, a dict of symbol → bars, and starting capital.

In [ ]:
INITIAL_CAPITAL = 100_000.0

strategy = BollingerBreakout(window=20, n_std=2.0)

results = qc.run_backtest(
    strategy=strategy,
    data={'ASSET': bars},
    initial_capital=INITIAL_CAPITAL,
)

equity_curve, ts = strip_sentinel(results)
returns          = calculate_returns(equity_curve)
metrics          = calculate_all_metrics(equity_curve, risk_free_rate=0.0)

#print(results)
print()
print(metrics)

## 5. Performance Tearsheet

In [ ]:
fig = plot_full_tearsheet(
    equity_curve,
    returns,
    timestamps=ts,
    title='Bollinger Band Breakout — Performance Tearsheet',
)
plt.show()

## 6. Visualizing the Signals

Before tuning parameters, it helps to see the bands on the price chart and verify the signals look right.

In [ ]:
window = 20
n_std  = 2.0

closes = np.array([b.close for b in bars])
means  = np.array([np.mean(closes[max(0, i - window):i]) if i >= window else np.nan
                   for i in range(1, len(closes) + 1)])
stds   = np.array([np.std(closes[max(0, i - window):i], ddof=0) if i >= window else np.nan
                   for i in range(1, len(closes) + 1)])

upper = means + n_std * stds
lower = means - n_std * stds
dates = pd.to_datetime(timestamps, unit='ns')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(dates, closes, linewidth=1.0, color='#2E86AB', label='Price', zorder=3)
ax.plot(dates, means,  linewidth=1.2, color='gray',    linestyle='--', label='Mean', alpha=0.7)
ax.fill_between(dates, lower, upper, alpha=0.15, color='orange', label='±2σ band')
ax.plot(dates, upper,  linewidth=0.8, color='orange', alpha=0.6)
ax.plot(dates, lower,  linewidth=0.8, color='orange', alpha=0.6)
ax.set_title('Bollinger Bands (window=20, n_std=2.0)', fontsize=14, fontweight='bold')
ax.set_ylabel('Price ($)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Parameter Sensitivity

The two main parameters are `window` (how long the lookback is) and `n_std` (how far out the bands sit). We sweep both and look at Sharpe ratio.

In [ ]:
windows    = [10, 15, 20, 30, 40, 60]
n_stds     = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5]

sharpe_grid = np.full((len(windows), len(n_stds)), np.nan)

for i, w in enumerate(windows):
    for j, n in enumerate(n_stds):
        s = BollingerBreakout(window=w, n_std=n)
        r = qc.run_backtest(strategy=s, data={'ASSET': bars}, initial_capital=INITIAL_CAPITAL)
        ec, _ = strip_sentinel(r)
        if len(ec) > 1:
            m = calculate_all_metrics(ec, risk_free_rate=0.0)
            sharpe_grid[i, j] = np.clip(m.sharpe_ratio, -3.0, 3.0)

fig, ax = plt.subplots(figsize=(11, 6))
im = ax.imshow(sharpe_grid, aspect='auto', cmap='RdYlGn', vmin=-1.5, vmax=2.0)

ax.set_xticks(range(len(n_stds)))
ax.set_xticklabels([str(n) for n in n_stds])
ax.set_yticks(range(len(windows)))
ax.set_yticklabels([str(w) for w in windows])
ax.set_xlabel('Band Width (n_std)', fontsize=12)
ax.set_ylabel('Lookback Window (bars)', fontsize=12)
ax.set_title('Sharpe Ratio — Parameter Sensitivity Heatmap', fontsize=14, fontweight='bold')

for i in range(len(windows)):
    for j in range(len(n_stds)):
        if not np.isnan(sharpe_grid[i, j]):
            color = 'white' if abs(sharpe_grid[i, j]) > 1.2 else 'black'
            ax.text(j, i, f'{sharpe_grid[i, j]:.2f}', ha='center', va='center',
                    fontsize=9, color=color, fontweight='bold')

plt.colorbar(im, ax=ax, label='Sharpe Ratio')
plt.tight_layout()
plt.show()

## 8. Walk-Forward Analysis

A grid search on a single train period will overfit. Walk-forward analysis splits the data into rolling windows: optimize on each in-sample window, then test on the next out-of-sample window.

This gives you a realistic picture of how parameter selection holds up when the market moves on.

In [ ]:
def grid_search(bars_list, windows, n_stds, initial_capital):
    """Return (window, n_std) with highest Sharpe on bars_list."""
    best_sharpe = -np.inf
    best_params = (20, 2.0)

    for w in windows:
        for n in n_stds:
            s = BollingerBreakout(window=w, n_std=n)
            r = qc.run_backtest(strategy=s, data={'ASSET': bars_list},
                                initial_capital=initial_capital)
            ec, _ = strip_sentinel(r)
            if len(ec) > 1:
                m = calculate_all_metrics(ec)
                if m.sharpe_ratio > best_sharpe:
                    best_sharpe = m.sharpe_ratio
                    best_params = (w, n)

    return best_params, best_sharpe


# walk-forward config
train_size = 400   # bars used to optimize
test_size  = 100   # bars used to evaluate out-of-sample
step       = 100   # slide forward by this many bars each fold

wf_windows = [15, 20, 30, 40]
wf_n_stds  = [1.5, 2.0, 2.5, 3.0]

oos_equity  = [INITIAL_CAPITAL]
oos_params  = []
fold_starts = []

start = 0
while start + train_size + test_size <= len(bars):
    train_bars = bars[start : start + train_size]
    test_bars  = bars[start + train_size : start + train_size + test_size]

    # optimize on train
    (best_w, best_n), _ = grid_search(
        train_bars,
        wf_windows, wf_n_stds, INITIAL_CAPITAL
    )

    # evaluate on test
    s   = BollingerBreakout(window=best_w, n_std=best_n)
    r   = qc.run_backtest(strategy=s, data={'ASSET': test_bars},
                          initial_capital=oos_equity[-1])
    ec, _ = strip_sentinel(r)

    if len(ec) > 1:
        oos_equity.extend(ec[1:].tolist())

    oos_params.append({'fold': len(oos_params) + 1, 'window': best_w,
                       'n_std': best_n, 'test_bars': test_size})
    fold_starts.append(start + train_size)
    start += step

oos_equity = np.array(oos_equity)
print(f'Walk-forward folds completed: {len(oos_params)}')
print()
print(pd.DataFrame(oos_params).to_string(index=False))

In [ ]:
# compare walk-forward equity vs buy-and-hold and single in-sample optimized run
bnh_results = qc.run_backtest(
    strategy=qc.BuyAndHold(),
    data={'ASSET': bars},
    initial_capital=INITIAL_CAPITAL,
)
bnh_equity, bnh_ts = strip_sentinel(bnh_results)

opt_strategy = BollingerBreakout(window=20, n_std=2.0)
opt_results  = qc.run_backtest(
    strategy=opt_strategy,
    data={'ASSET': bars},
    initial_capital=INITIAL_CAPITAL,
)
opt_equity, _ = strip_sentinel(opt_results)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(np.arange(len(bnh_equity)),  bnh_equity,  linewidth=1.5, color='gray',    label='Buy & Hold',                  alpha=0.8)
ax.plot(np.arange(len(opt_equity)),  opt_equity,  linewidth=1.5, color='#2E86AB', label='In-sample optimized (w=20, n=2.0)', linestyle='--')
ax.plot(np.arange(len(oos_equity)),  oos_equity,  linewidth=2.0, color='#27AE60', label='Walk-forward OOS')

for start_bar in fold_starts:
    ax.axvline(x=start_bar, color='orange', linestyle=':', linewidth=0.8, alpha=0.7)

ax.set_title('Walk-Forward Analysis vs. Baselines', fontsize=14, fontweight='bold')
ax.set_xlabel('Bar')
ax.set_ylabel('Portfolio Value ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

wf_metrics  = calculate_all_metrics(oos_equity)
opt_metrics = calculate_all_metrics(opt_equity)
bnh_metrics = calculate_all_metrics(bnh_equity)

comparison = pd.DataFrame({
    'Buy & Hold':           [f'{bnh_metrics.total_return:.1f}%', f'{bnh_metrics.sharpe_ratio:.2f}', f'{bnh_metrics.max_drawdown:.1f}%'],
    'In-sample optimized':  [f'{opt_metrics.total_return:.1f}%', f'{opt_metrics.sharpe_ratio:.2f}', f'{opt_metrics.max_drawdown:.1f}%'],
    'Walk-forward OOS':     [f'{wf_metrics.total_return:.1f}%',  f'{wf_metrics.sharpe_ratio:.2f}',  f'{wf_metrics.max_drawdown:.1f}%'],
}, index=['Total Return', 'Sharpe Ratio', 'Max Drawdown'])

print(comparison.to_string())

## 9. When Does This Break?

Bollinger Band breakouts fail in mean-reverting markets. If prices just oscillate around a mean without trending, every breakout signal is a false alarm, you buy the high and sell the low.

Let's verify this by running the same strategy on a mean-reverting Ornstein-Uhlenbeck process.

In [ ]:
np.random.seed(42)

# mean-reverting OU process
theta_ou = 0.08
mu_ou    = 100.0
ou_prices = np.empty(N)
ou_prices[0] = mu_ou
for i in range(1, N):
    ou_prices[i] = ou_prices[i - 1] + theta_ou * (mu_ou - ou_prices[i - 1]) + sigma * np.random.randn()

ou_bars = [
    qc.BarData('OU', timestamps[i], ou_prices[i],
               ou_prices[i] + 0.3, ou_prices[i] - 0.3,
               ou_prices[i], 1_000_000.0)
    for i in range(N)
]

# trending data (our original)
trend_results = qc.run_backtest(
    strategy=BollingerBreakout(window=20, n_std=2.0),
    data={'ASSET': bars},
    initial_capital=INITIAL_CAPITAL,
)

# mean-reverting data
ou_results = qc.run_backtest(
    strategy=BollingerBreakout(window=20, n_std=2.0),
    data={'OU': ou_bars},
    initial_capital=INITIAL_CAPITAL,
)

trend_ec, _ = strip_sentinel(trend_results)
ou_ec, _    = strip_sentinel(ou_results)

trend_m = calculate_all_metrics(trend_ec)
ou_m    = calculate_all_metrics(ou_ec)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(trend_ec, linewidth=1.5, color='#27AE60')
axes[0].set_title(f'Trending Market\nSharpe: {trend_m.sharpe_ratio:.2f} | Return: {trend_m.total_return:.1f}%',
                  fontsize=12, fontweight='bold')
axes[0].set_ylabel('Portfolio Value ($)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].grid(True, alpha=0.3)

axes[1].plot(ou_ec, linewidth=1.5, color='#E74C3C')
axes[1].set_title(f'Mean-Reverting Market\nSharpe: {ou_m.sharpe_ratio:.2f} | Return: {ou_m.total_return:.1f}%',
                  fontsize=12, fontweight='bold')
axes[1].set_ylabel('Portfolio Value ($)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].grid(True, alpha=0.3)

plt.suptitle('Bollinger Band Breakout: Trending vs. Mean-Reverting Market', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Trending market:')
print(f'  Sharpe: {trend_m.sharpe_ratio:.2f}  |  Return: {trend_m.total_return:.1f}%  |  Max DD: {trend_m.max_drawdown:.1f}%')
print()
print('Mean-reverting market:')
print(f'  Sharpe: {ou_m.sharpe_ratio:.2f}  |  Return: {ou_m.total_return:.1f}%  |  Max DD: {ou_m.max_drawdown:.1f}%')
print()
print('This is expected. Breakout strategies need trending markets.')
print('For mean-reverting markets, use MeanReversion. It bets on the opposite outcome.')

## Summary

Here's what we built and what to take away:

**The strategy:** Long on upper band breakout, short on lower band breakout, exits at the mean.

**What we learned:**
- `qc.Strategy` gives you a simple, consistent interface. Implement `on_data`, generate signals, let the engine handle the rest
- Walk-forward analysis reveals the gap between in-sample and out-of-sample performance. That gap is the real cost of parameter fitting
- Regime dependency is real: this strategy profits in trending markets and bleeds in mean-reverting ones. Any robust system needs to account for this, either by regime detection or by combining strategies with opposite sensitivities

**What to try next:**
- Add a trend filter (e.g., price > 200-day SMA) to skip signals in ranging markets
- Use `self.get_portfolio()` to scale position size with realized volatility
- Try this on two uncorrelated assets and compare portfolio-level Sharpe vs. single-asset
